In [45]:
import pandas as pd 
import torch
from plotnine import *
import numpy as np

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
df = pd.read_csv("mavoglurant_training_data_nlmixr.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
      .rename(
          columns={
              "ID": "id",
              "TIME": "time",
              "DV": "value",
              "DOSE": "dose"

          }
      )
      [["id", "time", "value", "dose", "WT"]]
      .astype({"value": "float"})
)
obs_df["output_name"] = "C15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

,ID,CMT,EVID,MDV,DV,AMT,TIME,DOSE,OCC,RATE,AGE,SEX,WT,HT
0,793,1,1,1,0.0,25.0,0.000,25.0,1,75,42,1,94.3,1.769997
1,793,2,0,0,491.0,0.0,0.200,25.0,1,0,42,1,94.3,1.769997
2,793,2,0,0,605.0,0.0,0.250,25.0,1,0,42,1,94.3,1.769997
3,793,2,0,0,556.0,0.0,0.367,25.0,1,0,42,1,94.3,1.769997
4,793,2,0,0,310.0,0.0,0.533,25.0,1,0,42,1,94.3,1.769997


,id,time,value,dose,WT,output_name,protocol_arm
1,793,0.200,491.0,25.0,94.3,C15,identity
2,793,0.250,605.0,25.0,94.3,C15,identity
3,793,0.367,556.0,25.0,94.3,C15,identity
4,793,0.533,310.0,25.0,94.3,C15,identity
5,793,0.700,237.0,25.0,94.3,C15,identity


In [47]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    path_to_solving_options="sv.json",
    inputs=["WT", "KbBR", "Clint","KbMU","KbAD","KbBO","KbRB" "dose"],
    outputs=["C15"],
)


struct_model = StructuralSimwork(model=model)

In [48]:
prior = {
    "model_intrinsic": {
        "KbBR": {"prior": np.exp(1.1)},
        "KbMU": {"prior": np.exp(0.3)},                
        "KbBO": {"prior": np.exp(0.03)},
        "KbAD": {"prior": np.exp(2)},
        "KbRB": {"prior": np.exp(0.3)}  
    },
    "pdu": {
        "CLint": {"prior": 7.6, "prior_omega": 4},
    },
    "error_model": {
        "C15": {"error_type": "additive", "sigma": 0.01},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_phase1_iterations=10,
        nb_phase2_iterations=10,
        plot_frames=5,
        optim_max_fun=2,
        verbose=False,
        mcmc_first_burn_in=0,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior, structural_model=struct_model, config=config
)

SchemaError: series 'id' contains duplicate values:
574     829
587     829
600     830
613     830
626     831
       ... 
2614    913
2627    914
2640    914
2653    915
2666    915
Name: id, Length: 156, dtype: object